# EndoScan AI — MRI Branch
> *Detecting endometriosis from pelvic MRI scans — EfficientNet-B0 fine-tuning.*

---

## CRISP-DM Roadmap

| Phase | Description | Status |
|---|---|---|
| 1. Business Understanding | Problem, stakeholders, success criteria | ✅ |
| 2. Data Understanding | UT-EndoMRI inspection, slice visualisation | ✅ |
| 3. Data Preparation | Consensus masks, extraction, splits, augmentation | ✅ |
| 4. Modelling | EfficientNet-B0 fine-tuning with early stopping | ← here |
| 5. Evaluation | AUC, sensitivity, specificity, Grad-CAM | upcoming |
| 6. Deployment | Gemini explanation layer + modality router | upcoming |

**Code philosophy:** Each cell defines reusable classes/functions.
Cell 8 is the only cell that *runs* the full pipeline — everything else is definitions.

## Cell 1 — Imports

In [1]:
import json, os, copy, time, random
from pathlib import Path

import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import rotate, zoom as scipy_zoom

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    classification_report, confusion_matrix
)
import timm

print('All imports OK')
print(f'PyTorch : {torch.__version__}')
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f'Device  : {device_name}')

All imports OK
PyTorch : 2.13.0+cu130
Device  : CPU


## Cell 2 — Config
> One place to change all paths and hyperparameters.

In [2]:
# ── Paths ──────────────────────────────────────────────────────────
MRI_ROOT        = Path.home() / 'Downloads/UT-EndoMRI'
PNG_DIR         = Path('./outputs/mri_slices')       # pre-extracted PNGs
RECORDS_PATH    = Path('./outputs/mri_slice_records.json')
CHECKPOINT_PATH = Path('./outputs/mri_best_model.pt')
RESULTS_DIR     = Path('./outputs')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Hyperparameters ────────────────────────────────────────────────
IMG_SIZE      = 224   # EfficientNet-B0 native input size
BATCH_SIZE    = 32    # increase to 64 on Colab GPU
EPOCHS        = 50    # max epochs — early stopping will kick in sooner
LR            = 1e-4
WEIGHT_DECAY  = 1e-4
PATIENCE      = 8     # early stopping patience
FREEZE_BLOCKS = 6     # freeze blocks 0-5, train block 6 + head (~28%)
DROPOUT       = 0.3
SEED          = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device         : {DEVICE}')
print(f'PNG dir exists : {PNG_DIR.exists()}')
print(f'Records exist  : {RECORDS_PATH.exists()}')

Device         : cpu
PNG dir exists : True
Records exist  : True


## Cell 3 — Preprocessing functions
NIfTI scanning, consensus mask generation, slice extraction, patient-level split.
These functions are only needed if `mri_slice_records.json` doesn't exist yet.

In [3]:
def scan_institution(inst_dir: Path) -> dict:
    """Scan D1_MHS or D2_TCPW and return a patient info dict."""
    patients = {}
    for pt_dir in sorted(inst_dir.iterdir()):
        if not pt_dir.is_dir():
            continue
        files  = list(pt_dir.glob('*.nii*'))
        seqs   = [f for f in files if any(
                  f.stem.endswith(s) for s in ['_T1','_T2','_T1FS','_T2FS'])]
        labels = [f for f in files if '_em' in f.stem]
        patients[pt_dir.name] = {
            'dir'      : pt_dir,
            'sequences': seqs,
            'em_labels': labels,
            'has_em'   : len(labels) > 0,
        }
    return patients


def resample_mask(mask: np.ndarray, target_shape: tuple) -> np.ndarray:
    """Nearest-neighbour resample mask to target_shape."""
    factors = [t / s for t, s in zip(target_shape, mask.shape)]
    return scipy_zoom(mask, factors, order=0)


def get_consensus_mask(pt_info: dict, target_shape: tuple,
                       threshold: int = 2):
    """
    Build a binary mask aligned to the T2 volume shape.
    D1 (3 raters): positive if >= threshold raters agree.
    D2 (1 rater) : positive if mask > 0.
    Returns float32 ndarray or None.
    """
    em_files = sorted([f for f in pt_info['em_labels'] if '_em_r' in f.stem])
    if not em_files:
        em_files = pt_info['em_labels']
    if not em_files:
        return None
    if len(em_files) == 1:
        m = nib.load(em_files[0]).get_fdata()
        if m.shape != target_shape:
            m = resample_mask(m, target_shape)
        return (m > 0).astype(np.float32)
    masks = []
    for f in em_files:
        m = nib.load(f).get_fdata()
        if m.shape != target_shape:
            m = resample_mask(m, target_shape)
        masks.append((m > 0).astype(np.float32))
    vote_map = np.zeros(target_shape, dtype=np.float32)
    for m in masks:
        vote_map += m
    return (vote_map >= threshold).astype(np.float32)


def extract_slices(patients: dict, institution: str,
                   is_d1: bool = True) -> list:
    """Label every axial T2 slice as positive (endo) or negative."""
    records = []
    for pt_id, info in patients.items():
        t2_files = [f for f in info['sequences'] if f.stem.endswith('_T2')]
        if not t2_files:
            print(f'  [skip] {pt_id} — no T2 found')
            continue
        t2_path  = t2_files[0]
        t2_vol   = nib.load(t2_path).get_fdata()
        t2_shape = t2_vol.shape
        mask = get_consensus_mask(info, t2_shape,
                                  threshold=2 if is_d1 else 1)
        for sl_idx in range(t2_shape[2]):
            label = int(mask is not None and mask[:, :, sl_idx].max() > 0)
            records.append({
                'patient_id' : pt_id,
                'institution': institution,
                'slice_idx'  : sl_idx,
                'label'      : label,
                't2_path'    : str(t2_path),
                'is_d1'      : is_d1,
            })
    return records


def patient_split(records: list, train_frac=0.70,
                  val_frac=0.15, seed=42):
    """
    Split by patient ID — never by slice — to prevent data leakage.
    Returns (train_records, val_records, test_records).
    """
    all_ids = list(set(r['patient_id'] for r in records))
    random.seed(seed)
    random.shuffle(all_ids)
    n_train = int(len(all_ids) * train_frac)
    n_val   = int(len(all_ids) * val_frac)
    train_ids = set(all_ids[:n_train])
    val_ids   = set(all_ids[n_train:n_train + n_val])
    test_ids  = set(all_ids[n_train + n_val:])
    train = [r for r in records if r['patient_id'] in train_ids]
    val   = [r for r in records if r['patient_id'] in val_ids]
    test  = [r for r in records if r['patient_id'] in test_ids]
    assert train_ids.isdisjoint(val_ids)   and \
           train_ids.isdisjoint(test_ids)  and \
           val_ids.isdisjoint(test_ids),   'Patient leak detected!'
    for name, recs in [('Train', train), ('Val', val), ('Test', test)]:
        n   = len(recs)
        pos = sum(r['label'] == 1 for r in recs)
        pts = len(set(r['patient_id'] for r in recs))
        print(f'  {name:5s}: {pts:3d} patients | {n:5d} slices | '
              f'{pos:4d} pos ({pos/n*100:.1f}%)')
    print('  No patient leaks across splits')
    return train, val, test


print('Preprocessing functions defined')

Preprocessing functions defined


## Cell 4 — Dataset & DataLoader
PNG-based loading is ~10x faster than NIfTI and avoids RAM crashes.
WeightedRandomSampler balances the 1:11 positive-to-negative imbalance.

In [4]:
def normalise_slice(sl: np.ndarray) -> np.ndarray:
    """Z-score normalise + clip ±3σ + scale to [-1, 1]."""
    sl   = sl.astype(np.float32)
    sl   = (sl - sl.mean()) / (sl.std() + 1e-8)
    return np.clip(sl, -3, 3) / 3.0


def augment_slice(sl: np.ndarray) -> np.ndarray:
    """Geometry-only augmentation — no colour (MRI intensity is clinical)."""
    if np.random.rand() > 0.5: sl = np.fliplr(sl)
    if np.random.rand() > 0.3: sl = np.flipud(sl)
    if np.random.rand() > 0.5:
        sl = rotate(sl, np.random.uniform(-15, 15), reshape=False, order=1)
    if np.random.rand() > 0.5:
        f  = np.random.uniform(0.9, 1.1)
        z  = scipy_zoom(sl, f, order=1)
        h, w   = sl.shape
        zh, zw = z.shape
        if f > 1.0:
            sl = z[(zh-h)//2:(zh-h)//2+h, (zw-w)//2:(zw-w)//2+w]
        else:
            ph, pw = (h-zh)//2, (w-zw)//2
            sl = np.pad(z, ((ph, h-zh-ph), (pw, w-zw-pw)), mode='constant')
    return sl.astype(np.float32)


class MRISliceDataset(Dataset):
    """
    Loads pre-extracted PNG slices — fast and memory-safe.
    PNG filename format: {patient_id}_sl{slice_idx:03d}_label{label}.png
    """
    def __init__(self, records, png_dir, augment=False, img_size=IMG_SIZE):
        self.records  = records
        self.png_dir  = Path(png_dir)
        self.augment  = augment
        self.img_size = img_size

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec   = self.records[idx]
        fname = (f"{rec['patient_id']}_"
                 f"sl{rec['slice_idx']:03d}_"
                 f"label{rec['label']}.png")
        sl = np.array(Image.open(self.png_dir / fname).convert('L'),
                      dtype=np.float32)
        sl = sl / 255.0 * 2.0 - 1.0      # [0,255] -> [-1, 1]
        if self.augment:
            sl = augment_slice(sl)
        if sl.shape != (self.img_size, self.img_size):
            sl = scipy_zoom(sl, (self.img_size / sl.shape[0],
                                  self.img_size / sl.shape[1]), order=1)
        sl_3ch = np.stack([sl, sl, sl], axis=0)   # greyscale -> 3-channel
        return (torch.tensor(sl_3ch, dtype=torch.float32),
                torch.tensor(rec['label'], dtype=torch.float32))


def make_dataloaders(train_recs, val_recs, test_recs,
                     png_dir, batch_size=BATCH_SIZE, num_workers=0):
    """Build train/val/test DataLoaders with balanced weighted sampling."""
    train_ds = MRISliceDataset(train_recs, png_dir, augment=True)
    val_ds   = MRISliceDataset(val_recs,   png_dir, augment=False)
    test_ds  = MRISliceDataset(test_recs,  png_dir, augment=False)

    # Weighted sampler — balances pos/neg in every training batch
    labels  = [r['label'] for r in train_recs]
    n_pos   = sum(labels)
    n_neg   = len(labels) - n_pos
    weights = [1/n_pos if l == 1 else 1/n_neg for l in labels]
    sampler = WeightedRandomSampler(
        torch.tensor(weights, dtype=torch.float32),
        len(weights), replacement=True
    )

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              sampler=sampler, num_workers=num_workers)
    val_loader   = DataLoader(val_ds, batch_size=batch_size,
                              shuffle=False, num_workers=num_workers)
    test_loader  = DataLoader(test_ds, batch_size=batch_size,
                              shuffle=False, num_workers=num_workers)

    print(f'Train : {len(train_loader)} batches ({len(train_ds)} samples)')
    print(f'Val   : {len(val_loader)} batches ({len(val_ds)} samples)')
    print(f'Test  : {len(test_loader)} batches ({len(test_ds)} samples)')
    return train_loader, val_loader, test_loader


print('Dataset & DataLoader defined')

Dataset & DataLoader defined


## Cell 5 — Model builder
EfficientNet-B0 with frozen early layers. ~28% trainable — optimal for small medical datasets.

In [5]:
def build_model(dropout=DROPOUT, freeze_blocks=FREEZE_BLOCKS,
                pretrained=True):
    """
    EfficientNet-B0 pretrained on ImageNet.
    Freezes blocks 0 to freeze_blocks-1.
    Replaces classifier with a single binary output (BCEWithLogitsLoss).
    """
    model = timm.create_model('efficientnet_b0', pretrained=pretrained)

    for param in model.parameters():
        param.requires_grad = False

    for block in model.blocks[freeze_blocks:]:
        for param in block.parameters():
            param.requires_grad = True

    for m in [model.conv_head, model.bn2]:
        for param in m.parameters():
            param.requires_grad = True

    in_feat = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_feat, 1)
    )

    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Total parameters     : {total:,}')
    print(f'Trainable parameters : {trainable:,} ({trainable/total*100:.1f}%)')
    return model


print('Model builder defined')

Model builder defined


## Cell 6 — Training engine
Early stopping, model checkpointing, cosine LR schedule, AUC tracking per epoch.

In [ ]:
class EarlyStopping:
    """
    Stop training when val loss stops improving.
    patience  : epochs to wait after last improvement.
    min_delta : minimum change to count as improvement.
    """
    def __init__(self, patience=PATIENCE, min_delta=1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best_loss  = float('inf')
        self.triggered  = False

    def __call__(self, val_loss: float) -> bool:
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter   = 0
        else:
            self.counter += 1
            print(f'  EarlyStopping counter: {self.counter}/{self.patience}')
            if self.counter >= self.patience:
                self.triggered = True
        return self.triggered


def train_one_epoch(model, loader, criterion, optimizer, device):
    """One full pass over the training set. Returns loss and accuracy."""
    model.train()
    total_loss = correct = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device).unsqueeze(1)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        preds       = (torch.sigmoid(out) > 0.5).float()
        correct    += (preds == labels).sum().item()
        total      += imgs.size(0)
    return {'loss': total_loss / total, 'acc': correct / total * 100}


# ── ADD this to Cell 6 — updated evaluate() function ──────────────

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """Evaluate on val/test. Returns loss, accuracy, AUC, precision, recall, F1."""
    model.eval()
    total_loss = correct = total = 0
    all_probs, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device).unsqueeze(1)
        out   = model(imgs)
        loss  = criterion(out, labels)
        probs = torch.sigmoid(out).cpu().squeeze(1)
        total_loss += loss.item() * imgs.size(0)
        preds       = (probs > 0.5).float()
        correct    += (preds == labels.cpu().squeeze(1)).sum().item()
        total      += imgs.size(0)
        all_probs.extend(probs.numpy())
        all_labels.extend(labels.cpu().squeeze(1).numpy())

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    all_preds  = (all_probs > 0.5).astype(float)

    try:    auc = roc_auc_score(all_labels, all_probs)
    except: auc = float('nan')

    tp = ((all_preds == 1) & (all_labels == 1)).sum()
    fp = ((all_preds == 1) & (all_labels == 0)).sum()
    fn = ((all_preds == 0) & (all_labels == 1)).sum()
    tn = ((all_preds == 0) & (all_labels == 0)).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else float('nan')
    recall    = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
    f1        = (2 * precision * recall / (precision + recall)
                 if not (np.isnan(precision) or np.isnan(recall) or
                         precision + recall == 0) else float('nan'))
    specificity = tn / (tn + fp) if (tn + fp) > 0 else float('nan')

    return {
        'loss': total_loss/total, 'acc': correct/total*100,
        'auc': auc, 'precision': precision, 'recall': recall,
        'f1': f1, 'specificity': specificity,
        'probs': all_probs, 'labels': all_labels
    }

def train(model, train_loader, val_loader, pos_weight, device,
          epochs=EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY,
          patience=PATIENCE, checkpoint_path=CHECKPOINT_PATH):
    """
    Full training loop with:
      - Early stopping on val loss
      - Automatic checkpoint saving on every improvement
      - Cosine annealing LR schedule
      - AUC tracked and printed every epoch
    """
    criterion  = nn.BCEWithLogitsLoss(
                     pos_weight=torch.tensor([pos_weight]).to(device))
    optimizer  = Adam(filter(lambda p: p.requires_grad, model.parameters()),
                      lr=lr, weight_decay=weight_decay)
    scheduler  = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    stopper    = EarlyStopping(patience=patience)
    best_w     = copy.deepcopy(model.state_dict())
    history = {'train_loss':[], 'val_loss':[], 'train_acc':[],
           'val_acc':[], 'val_auc':[], 'val_recall':[],
           'val_precision':[], 'val_f1':[], 'val_specificity':[]}

    print(f'Training | max {epochs} epochs | patience={patience} | '
          f'pos_weight={pos_weight:.2f}')
    print('-' * 72)

    for epoch in range(1, epochs + 1):
        t0      = time.time()
        train_m = train_one_epoch(model, train_loader, criterion,
                                   optimizer, device)
        val_m   = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        history['val_recall'].append(val_m['recall'])
        history['val_precision'].append(val_m['precision'])
        history['val_f1'].append(val_m['f1'])
        history['val_specificity'].append(val_m['specificity'])

        improved = val_m['loss'] < stopper.best_loss - stopper.min_delta
        tag      = '  <- best' if improved else ''
        print(f'Epoch {epoch:3d}/{epochs} | '
            f'loss {train_m["loss"]:.4f} | '
            f'val loss {val_m["loss"]:.4f} '
            f'acc {val_m["acc"]:.1f}% '
            f'auc {val_m["auc"]:.3f} '
            f'rec {val_m["recall"]:.3f} '
            f'f1 {val_m["f1"]:.3f} | '
            f'{time.time()-t0:.0f}s{tag}')
        if improved:
            best_w = copy.deepcopy(model.state_dict())
            torch.save({'epoch': epoch,
                        'model_state_dict': best_w,
                        'val_loss': val_m['loss'],
                        'val_auc':  val_m['auc'],
                        'history':  history},
                       checkpoint_path)

        if stopper(val_m['loss']):
            print(f'Early stopping at epoch {epoch}.')
            break

    model.load_state_dict(best_w)
    best_ep = history['val_loss'].index(min(history['val_loss'])) + 1
    print(f'\nDone. Best epoch: {best_ep} | '
          f'val loss: {min(history["val_loss"]):.4f} | '
          f'val AUC: {max(history["val_auc"]):.4f}')
    return history


print('Training engine defined')

Training engine defined


## Cell 7 — Evaluation & plotting functions

In [7]:
@torch.no_grad()
def evaluate_test_set(model, loader, device, threshold=0.5):
    """Full test set evaluation — AUC, sensitivity, specificity, report."""
    model.eval()
    all_probs, all_labels = [], []
    for imgs, labels in loader:
        probs = torch.sigmoid(model(imgs.to(device))).cpu().squeeze(1)
        all_probs.extend(probs.numpy())
        all_labels.extend(labels.numpy())
    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    all_preds  = (all_probs > threshold).astype(float)

    try:    auc = roc_auc_score(all_labels, all_probs)
    except: auc = float('nan')

    cm = confusion_matrix(all_labels, all_preds)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
        spec = tn / (tn + fp) if (tn + fp) > 0 else float('nan')
    else:
        sens = spec = float('nan')

    try:
        report = classification_report(
            all_labels, all_preds,
            target_names=['No endo', 'Endo'], digits=3)
    except ValueError as e:
        report = str(e)

    return {'auc': auc, 'accuracy': (all_preds == all_labels).mean() * 100,
            'sensitivity': sens, 'specificity': spec, 'cm': cm,
            'probs': all_probs, 'preds': all_preds,
            'labels': all_labels, 'report': report}


def print_test_results(metrics, modality='MRI'):
    print('=' * 55)
    print(f'  TEST SET EVALUATION — {modality}')
    print('=' * 55)
    n = len(metrics['labels'])
    pos = int(metrics['labels'].sum())
    print(f'  Samples      : {n} ({pos} pos / {n-pos} neg)')
    print(f'  AUC-ROC      : {metrics["auc"]:.4f}')
    print(f'  Accuracy     : {metrics["accuracy"]:.1f}%')
    print(f'  Sensitivity  : {metrics["sensitivity"]:.3f}  (recall for endo)')
    print(f'  Specificity  : {metrics["specificity"]:.3f}  (recall for no endo)')
    print()
    print(metrics['report'])
    print('=' * 55)


def plot_results(metrics, history, modality='MRI', save_path=None):
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))

    axes[0].plot(history['train_loss'], label='Train')
    axes[0].plot(history['val_loss'],   label='Val')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(history['train_acc'], label='Train')
    axes[1].plot(history['val_acc'],   label='Val')
    axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)

    try:
        fpr, tpr, _ = roc_curve(metrics['labels'], metrics['probs'])
        axes[2].plot(fpr, tpr, lw=2, label=f'AUC={metrics["auc"]:.3f}')
        axes[2].plot([0,1],[0,1],'--',color='gray')
        axes[2].set_title('ROC'); axes[2].legend(); axes[2].grid(alpha=0.3)
    except Exception:
        axes[2].set_title('ROC — insufficient classes')

    pos_p = metrics['probs'][metrics['labels'] == 1]
    neg_p = metrics['probs'][metrics['labels'] == 0]
    axes[3].hist(neg_p, bins=15, alpha=0.6, label='No endo', color='steelblue')
    axes[3].hist(pos_p, bins=15, alpha=0.6, label='Endo',    color='coral')
    axes[3].axvline(0.5, color='red', linestyle='--', label='Threshold')
    axes[3].set_title('Confidence'); axes[3].legend(); axes[3].grid(alpha=0.3)

    plt.suptitle(f'EfficientNet-B0 — {modality}', fontsize=12)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Plot saved -> {save_path}')
    plt.show()


print('Evaluation functions defined')

Evaluation functions defined


## Cell 8 — Run pipeline
This is the **only cell that executes the pipeline**.
All logic lives in the functions above — this just calls them in order.

In [8]:
# ── 1. Load or build slice records ─────────────────────────────────
if RECORDS_PATH.exists():
    with open(RECORDS_PATH) as f:
        all_records = json.load(f)
    print(f'Loaded {len(all_records)} records from {RECORDS_PATH}')
else:
    print('Records not found — building from NIfTI...')
    d1_patients = scan_institution(MRI_ROOT / 'D1_MHS')
    d2_patients = scan_institution(MRI_ROOT / 'D2_TCPW')
    d1_records  = extract_slices(d1_patients, 'D1_MHS', is_d1=True)
    d2_records  = extract_slices(d2_patients, 'D2_TCPW', is_d1=False)
    all_records = d1_records + d2_records
    with open(RECORDS_PATH, 'w') as f:
        json.dump(all_records, f, indent=2)
    print(f'Saved {len(all_records)} records -> {RECORDS_PATH}')

# ── 2. Patient-level split ─────────────────────────────────────────
print('\nPatient-level split:')
train_records, val_records, test_records = patient_split(
    all_records, seed=SEED)

# ── 3. Class weights ───────────────────────────────────────────────
train_labels = [r['label'] for r in train_records]
n_pos        = sum(train_labels)
n_neg        = len(train_labels) - n_pos
pos_weight   = n_neg / n_pos
print(f'\npos_weight = {pos_weight:.2f}  (n_pos={n_pos}, n_neg={n_neg})')

# ── 4. DataLoaders ─────────────────────────────────────────────────
print('\nDataLoaders:')
train_loader, val_loader, test_loader = make_dataloaders(
    train_records, val_records, test_records,
    png_dir=PNG_DIR, batch_size=BATCH_SIZE
)

# ── 5. Build model ─────────────────────────────────────────────────
print('\nModel:')
model = build_model().to(DEVICE)

# ── 6. Train ───────────────────────────────────────────────────────
history = train(
    model, train_loader, val_loader,
    pos_weight      = pos_weight,
    device          = DEVICE,
    epochs          = EPOCHS,
    lr              = LR,
    weight_decay    = WEIGHT_DECAY,
    patience        = PATIENCE,
    checkpoint_path = CHECKPOINT_PATH,
)

# ── 7. Evaluate ────────────────────────────────────────────────────
metrics = evaluate_test_set(model, test_loader, DEVICE)
print_test_results(metrics, modality='MRI')
plot_results(metrics, history, modality='MRI',
             save_path=str(RESULTS_DIR / 'mri_results.png'))

Loaded 4242 records from outputs/mri_slice_records.json

Patient-level split:
  Train:  64 patients |  2918 slices |  241 pos (8.3%)
  Val  :  13 patients |   649 slices |   73 pos (11.2%)
  Test :  15 patients |   675 slices |   38 pos (5.6%)
  No patient leaks across splits

pos_weight = 11.11  (n_pos=241, n_neg=2677)

DataLoaders:
Train : 92 batches (2918 samples)
Val   : 21 batches (649 samples)
Test  : 22 batches (675 samples)

Model:


Total parameters     : 4,008,829
Trainable parameters : 1,130,673 (28.2%)
Training | max 50 epochs | patience=8 | pos_weight=11.11
------------------------------------------------------------------------
Epoch   1/50 | train loss 2.1738 acc 57.0% | val loss 1.3790 acc 35.4% auc 0.737 | 188s  <- best


KeyboardInterrupt: 

## Cell 9 — Save final model with full metadata

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'history'         : history,
    'train_patients'  : list(set(r['patient_id'] for r in train_records)),
    'val_patients'    : list(set(r['patient_id'] for r in val_records)),
    'test_patients'   : list(set(r['patient_id'] for r in test_records)),
    'metrics': {
        'auc'        : metrics['auc'],
        'accuracy'   : metrics['accuracy'],
        'sensitivity': metrics['sensitivity'],
        'specificity': metrics['specificity'],
    },
    'config': {
        'img_size'     : IMG_SIZE,
        'batch_size'   : BATCH_SIZE,
        'epochs_run'   : len(history['val_loss']),
        'lr'           : LR,
        'freeze_blocks': FREEZE_BLOCKS,
        'dropout'      : DROPOUT,
        'modality'     : 'MRI',
    }
}, CHECKPOINT_PATH)

print(f'Model saved       -> {CHECKPOINT_PATH}')
print(f'AUC-ROC           : {metrics["auc"]:.4f}')
print(f'Sensitivity       : {metrics["sensitivity"]:.3f}')
print(f'Specificity       : {metrics["specificity"]:.3f}')
print(f'Epochs trained    : {len(history["val_loss"])}')